In [1]:
%pip install -q -U transformers accelerate peft bitsandbytes datasets


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import json

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel


In [3]:
if torch.cuda.is_available():
    device = "cuda"
    print("GPU:", torch.cuda.get_device_name(0))
    major, minor = torch.cuda.get_device_capability(0)
    USE_BF16 = major >= 8
    print(f"Compute capability: {major}.{minor} -> using {'bf16' if USE_BF16 else 'fp16'}")
else:
    device = "cpu"
    USE_BF16 = False
    print("CUDA not available -- running on CPU (will be slow).")

print("Selected device:", device)


GPU: NVIDIA GeForce RTX 4050 Laptop GPU
Compute capability: 8.9 -> using bf16
Selected device: cuda


In [20]:
BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_DIR = "./qwen25-1_5b-ocr-json-lora/final_adapter"   # change if saved elsewhere
VAL_PATH = "./val_unseen.jsonl"                                     # change if val.jsonl is elsewhere
MAX_NEW_TOKENS = 3000

# If True, also evaluates the untrained base model on the same set for comparison.
# Doubles runtime -- set False if you only want fine-tuned numbers.
ALSO_EVAL_BASE_MODEL = True


In [19]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if device == "cuda":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.float32,
    ).to(device)

finetuned_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
finetuned_model.eval()
print("Fine-tuned model loaded.")


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Fine-tuned model loaded.


In [21]:
val_examples = []
with open(VAL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        val_examples.append(json.loads(line))

print(f"Loaded {len(val_examples)} validation examples.")
print("\nExample:")
print(val_examples[0]["messages"][1]["content"][:300])


Loaded 100 validation examples.

Example:
Extract fields into JSON: {"vendor": string, "invoice_no": string, "date": string, "total": float, "items": [{ "desc": string, "amount": float }]}

RAW OCR INPUT:
Please find below the billing summary from Ironclad Hardware Ltd.
Ref No 3539572182, dated 20250420.
Charges:
  - Corrugated Box Large: R


In [22]:
def generate_response(model, tokenizer, messages, max_new_tokens=MAX_NEW_TOKENS):
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


def extract_json_block(text):
    """Pull JSON out of a ```json ... ``` fence, or try the raw text as a fallback."""
    match = re.search(r"```json\s*(.*?)\s*```", text, re.DOTALL)
    candidate = match.group(1) if match else text.strip()
    try:
        return json.loads(candidate), None
    except Exception as e:
        return None, str(e)


def normalize_amount(x):
    try:
        return round(float(x), 2)
    except (TypeError, ValueError):
        return None


def normalize_str(x):
    if x is None:
        return ""
    return re.sub(r"\s+", " ", str(x)).strip().lower()


In [23]:
def compare_record(expected, predicted):
    """Returns a dict of per-field booleans (True = correct) and diagnostic info.
    predicted may be None if JSON parsing failed entirely."""
    result = {
        "json_valid": predicted is not None,
        "vendor_match": False,
        "invoice_no_match": False,
        "date_match": False,
        "total_match": False,
        "items_match": False,
        "items_precision": None,
        "items_recall": None,
    }

    if predicted is None:
        return result

    result["vendor_match"] = normalize_str(expected.get("vendor")) == normalize_str(predicted.get("vendor"))
    result["invoice_no_match"] = normalize_str(expected.get("invoice_no")) == normalize_str(predicted.get("invoice_no"))
    result["date_match"] = normalize_str(expected.get("date")) == normalize_str(predicted.get("date"))

    exp_total = normalize_amount(expected.get("total"))
    pred_total = normalize_amount(predicted.get("total"))
    result["total_match"] = exp_total is not None and exp_total == pred_total

    exp_items = expected.get("items", [])
    pred_items = predicted.get("items", []) if isinstance(predicted.get("items"), list) else []

    exp_set = set()
    for it in exp_items:
        amt = normalize_amount(it.get("amount"))
        exp_set.add((normalize_str(it.get("desc")), amt))

    pred_set = set()
    for it in pred_items:
        if not isinstance(it, dict):
            continue
        amt = normalize_amount(it.get("amount"))
        pred_set.add((normalize_str(it.get("desc")), amt))

    if exp_set:
        overlap = exp_set & pred_set
        result["items_recall"] = len(overlap) / len(exp_set)
    if pred_set:
        overlap = exp_set & pred_set
        result["items_precision"] = len(overlap) / len(pred_set)

    result["items_match"] = exp_set == pred_set

    return result


In [24]:
def run_evaluation(model, label):
    print(f"\n{'='*70}\nEvaluating: {label}\n{'='*70}")
    all_results = []
    failures = []

    for i, example in enumerate(val_examples):
        messages = example["messages"]
        system_msg, user_msg, assistant_msg = messages[0], messages[1], messages[2]

        # Ground truth: parse the assistant's fenced JSON
        expected, _ = extract_json_block(assistant_msg["content"])
        assert expected is not None, f"Ground truth JSON in val.jsonl example {i} failed to parse!"

        # Model prediction: run inference on system+user only
        raw_output = generate_response(model, tokenizer, [system_msg, user_msg])
        predicted, parse_err = extract_json_block(raw_output)

        comp = compare_record(expected, predicted)
        comp["index"] = i
        all_results.append(comp)

        if not comp["json_valid"] or not (comp["vendor_match"] and comp["invoice_no_match"]
                                           and comp["date_match"] and comp["total_match"]
                                           and comp["items_match"]):
            failures.append({
                "index": i,
                "raw_output": raw_output,
                "expected": expected,
                "predicted": predicted,
                "parse_err": parse_err,
                "comparison": comp,
            })

        if (i + 1) % 10 == 0 or (i + 1) == len(val_examples):
            print(f"  Processed {i+1}/{len(val_examples)}")

    return all_results, failures


In [25]:
ft_results, ft_failures = run_evaluation(finetuned_model, "Fine-tuned model")



Evaluating: Fine-tuned model
  Processed 10/100
  Processed 20/100
  Processed 30/100
  Processed 40/100
  Processed 50/100
  Processed 60/100
  Processed 70/100
  Processed 80/100
  Processed 90/100
  Processed 100/100


In [27]:
if ALSO_EVAL_BASE_MODEL:
    # Reuse the same underlying base_model weights but disable the LoRA adapter
    # to get untrained behavior without reloading from scratch.
    with finetuned_model.disable_adapter():
        base_results, base_failures = run_evaluation(finetuned_model, "Base model (adapter disabled)")
else:
    base_results, base_failures = None, None



Evaluating: Base model (adapter disabled)


KeyboardInterrupt: 

In [28]:
def summarize(results, label):
    n = len(results)
    def rate(key):
        return sum(1 for r in results if r[key]) / n

    exact_match = sum(
        1 for r in results
        if r["json_valid"] and r["vendor_match"] and r["invoice_no_match"]
        and r["date_match"] and r["total_match"] and r["items_match"]
    ) / n

    items_recalls = [r["items_recall"] for r in results if r["items_recall"] is not None]
    items_precisions = [r["items_precision"] for r in results if r["items_precision"] is not None]

    print(f"\n{'='*70}\n{label} -- Summary over {n} validation examples\n{'='*70}")
    print(f"  JSON validity rate     : {rate('json_valid'):.1%}")
    print(f"  Vendor match rate      : {rate('vendor_match'):.1%}")
    print(f"  Invoice No. match rate : {rate('invoice_no_match'):.1%}")
    print(f"  Date match rate        : {rate('date_match'):.1%}")
    print(f"  Total match rate       : {rate('total_match'):.1%}")
    print(f"  Items exact-set match  : {rate('items_match'):.1%}")
    if items_recalls:
        print(f"  Items avg recall       : {sum(items_recalls)/len(items_recalls):.1%}")
    if items_precisions:
        print(f"  Items avg precision    : {sum(items_precisions)/len(items_precisions):.1%}")
    print(f"  FULL RECORD exact match: {exact_match:.1%}")
    return {
        "json_validity": rate("json_valid"),
        "vendor": rate("vendor_match"),
        "invoice_no": rate("invoice_no_match"),
        "date": rate("date_match"),
        "total": rate("total_match"),
        "items_exact": rate("items_match"),
        "full_exact_match": exact_match,
    }

ft_summary = summarize(ft_results, "FINE-TUNED MODEL")
if ALSO_EVAL_BASE_MODEL:
    base_summary = summarize(base_results, "BASE MODEL (untrained)")



FINE-TUNED MODEL -- Summary over 100 validation examples
  JSON validity rate     : 99.0%
  Vendor match rate      : 91.0%
  Invoice No. match rate : 51.0%
  Date match rate        : 81.0%
  Total match rate       : 82.0%
  Items exact-set match  : 25.0%
  Items avg recall       : 58.7%
  Items avg precision    : 58.7%
  FULL RECORD exact match: 9.0%

BASE MODEL (untrained) -- Summary over 100 validation examples
  JSON validity rate     : 93.0%
  Vendor match rate      : 66.0%
  Invoice No. match rate : 60.0%
  Date match rate        : 11.0%
  Total match rate       : 67.0%
  Items exact-set match  : 28.0%
  Items avg recall       : 62.0%
  Items avg precision    : 62.0%
  FULL RECORD exact match: 0.0%


In [29]:
if ALSO_EVAL_BASE_MODEL:
    metrics = ["json_validity", "vendor", "invoice_no", "date", "total", "items_exact", "full_exact_match"]
    print(f"{'Metric':<20} {'Base':>10} {'Fine-tuned':>12} {'Delta':>10}")
    print("-" * 54)
    for m in metrics:
        b = base_summary[m]
        f = ft_summary[m]
        print(f"{m:<20} {b:>9.1%} {f:>11.1%} {f-b:>+9.1%}")
else:
    print("Set ALSO_EVAL_BASE_MODEL = True above to see a base-vs-fine-tuned comparison table.")


Metric                     Base   Fine-tuned      Delta
------------------------------------------------------
json_validity            93.0%       99.0%     +6.0%
vendor                   66.0%       91.0%    +25.0%
invoice_no               60.0%       51.0%     -9.0%
date                     11.0%       81.0%    +70.0%
total                    67.0%       82.0%    +15.0%
items_exact              28.0%       25.0%     -3.0%
full_exact_match          0.0%        9.0%     +9.0%


In [30]:
print(f"Fine-tuned model: {len(ft_failures)} / {len(val_examples)} examples had at least one field wrong.\n")

N_TO_SHOW = 5
for f in ft_failures[:N_TO_SHOW]:
    print("=" * 70)
    print(f"Example index: {f['index']}")
    print("-" * 70)
    print("RAW MODEL OUTPUT:")
    print(f["raw_output"])
    print("-" * 70)
    print("EXPECTED:")
    print(json.dumps(f["expected"], indent=2))
    print("-" * 70)
    print("PREDICTED (parsed, or None if invalid JSON):")
    print(json.dumps(f["predicted"], indent=2) if f["predicted"] else f"INVALID JSON -- {f['parse_err']}")
    print("-" * 70)
    print("Field-level comparison:", f["comparison"])
    print()

if len(ft_failures) > N_TO_SHOW:
    print(f"... and {len(ft_failures) - N_TO_SHOW} more failures not shown. "
          f"Increase N_TO_SHOW above to see more.")


Fine-tuned model: 91 / 100 examples had at least one field wrong.

Example index: 0
----------------------------------------------------------------------
RAW MODEL OUTPUT:
```json
{
  "vendor": "Ironclad Hardware Ltd",
  "invoice_no": "3539572182",
  "date": "2025-04-20",
  "total": 230739.01,
  "items": [
    {
      "desc": "Corrugated Box 5m",
      "amount": 54794.47
    },
    {
      "desc": "Wooden Pallet Std",
      "amount": 57812.13
    },
    {
      "desc": "Galvanized Nut Bolt Set",
      "amount": 56330.68
    },
    {
      "desc": "Thermal Insulation Roll",
      "amount": 43995.27
    },
    {
      "desc": "Rubber Matting Roll",
      "amount": 18006.46
    }
  ]
}
```
----------------------------------------------------------------------
EXPECTED:
{
  "vendor": "Ironclad Hardware Ltd",
  "invoice_no": "3539572182",
  "date": "2025-04-20",
  "total": 230739.01,
  "items": [
    {
      "desc": "Corrugated Box Large",
      "amount": 54794.47
    },
    {
      "desc"

In [ ]:
import csv

RESULTS_CSV = "./val_results_finetuned.csv"

with open(RESULTS_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "index", "json_valid", "vendor_match", "invoice_no_match",
        "date_match", "total_match", "items_match", "items_precision", "items_recall",
    ])
    writer.writeheader()
    for r in ft_results:
        writer.writerow(r)

print(f"Wrote per-example results to {RESULTS_CSV}")

Wrote per-example results to ./val_results_finetuned.csv
